# Vaccum

In [ ]:
import boto3
import pprint
import sys
from botocore.client import Config
from botocore.exceptions import ClientError
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import *

MINIO_ENDPOINT = "http://minio:9000"
MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
DATABASE = "default"
BUCKET_BRONZE = "bronze"
BUCKET_SILVER = "silver"
BUCKET_GOLD = "gold"

In [ ]:
%%time
spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("MyAppM5Class03") \
    .config("spark.eventLog.enabled", "true") \
    .config("spark.eventLog.dir", "file:/home/jovyan/work/spark-logs") \
    .config("spark.history.fs.logDirectory", "file:/home/jovyan/work/spark-logs") \
    .config("log4j.rootCategory", "INFO, console") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.driver.memory", "1536m") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.storage.memoryFraction", "0.4") \
    .config("spark.shuffle.memoryFraction", "0.5") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "512m") \
    .config("spark.sql.parquet.compression.codec", "gzip") \
    .config("spark.sql.orc.compression.codec", "zlib") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .config("spark.cleaner.referenceTracking.cleanCheckpoints", "true") \
    .config("spark.executor.cleanupOnShutdown", "true") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

In [ ]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'  # Can be any valid region
)

## PROJECT 1: HOTEL BOOKING

- [Data source](https://www.kaggle.com/datasets/mojtaba142/hotel-booking)

Let's explore Vaccum

1. Reading data from raw from **Staging**
2. Write data into bronze
3. Changing TBLPROPERTIES for the delta table and controlling Vaccum behavior
4. Operation: **Update**
5. Apply Vaccum in the Delta Table **WITH** TBLPROPERTIES
6. Apply Vaccum in the Delta Table **WITHOUT** TBLPROPERTIES


### 1 - Reading data from raw from **Staging**

In [ ]:
## Functions: utils

def build_arrival_partition_date(df):
    df = df.withColumn(
        "arrival_date_day_of_month_padded",
        F.lpad("arrival_date_day_of_month", 2, "0")
    )

    df = df.withColumn(
        "arrival_date_month_number",
        F.when(F.col("arrival_date_month") == "January", "01")
        .when(F.col("arrival_date_month") == "February", "02")
        .when(F.col("arrival_date_month") == "March", "03")
        .when(F.col("arrival_date_month") == "April", "04")
        .when(F.col("arrival_date_month") == "May", "05")
        .when(F.col("arrival_date_month") == "June", "06")
        .when(F.col("arrival_date_month") == "July", "07")
        .when(F.col("arrival_date_month") == "August", "08")
        .when(F.col("arrival_date_month") == "September", "09")
        .when(F.col("arrival_date_month") == "October", "10")
        .when(F.col("arrival_date_month") == "November", "11")
        .when(F.col("arrival_date_month") == "December", "12")
    )
    df = (
        df.withColumn("arrival_partition_date", 
          F.to_date(
            F.concat_ws(
                "-", 
                df.arrival_date_year, 
                df.arrival_date_month_number,
                df.arrival_date_day_of_month_padded
            ), 
              "yyyy-MM-dd"
          )
        )
    )
    
    return df.drop("arrival_date_day_of_month_padded", "arrival_date_month_number")

def list_files_minio(bucket_name, prefix):
    """
    List all files in a given MinIO bucket and prefix using boto3.

    :param bucket_name: Name of the bucket
    :param prefix: Prefix path within the bucket
    :return: List of file keys
    """
    try:
        paginator = s3_client.get_paginator('list_objects_v2')
        pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)
        files = []
        for page in pages:
            if 'Contents' in page:
                for obj in page['Contents']:
                    files.append(obj['Key'])
        return files
    except ClientError as e:
        print(f"Error listing files in bucket '{bucket_name}' with prefix '{prefix}': {e}")
        return []

def apply_vacuum_and_list_files(bucket_name, table_prefix, retention_hours):
    """
    List files before VACUUM, apply VACUUM, and list files after VACUUM.

    :param bucket_name: Name of the MinIO bucket
    :param table_prefix: Prefix path to the Delta table within the bucket
    :param retention_hours: Retention period in hours for VACUUM
    :param s3_client: boto3 S3 client
    :param spark: SparkSession object
    """
    print("\n===== Listing Files Before VACUUM =====")
    files_before = list_files_minio(bucket_name, table_prefix)
    print(f"Total files before VACUUM: {len(files_before)}")
    print("Sample files before VACUUM:")
    pprint.pprint(files_before[:10])

    # Path to the Delta table using S3A protocol
    delta_table_s3_path = f"s3a://{bucket_name}/{table_prefix}"

    # Initialize DeltaTable
    try:
        delta_table = DeltaTable.forPath(spark, delta_table_s3_path)
    except Exception as e:
        print(f"Error initializing DeltaTable at path '{delta_table_s3_path}': {e}")
        raise

    # Perform VACUUM
    print(f"\n===== Applying VACUUM with Retention of {retention_hours} Hours =====")
    try:
        delta_table.vacuum(retentionHours=retention_hours)
        print("VACUUM completed successfully.")
    except Exception as e:
        print(f"Error during VACUUM: {e}")
        raise

    # List files after VACUUM
    print("\n===== Listing Files After VACUUM =====")
    files_after = list_files_minio(bucket_name, table_prefix)
    print(f"Total files after VACUUM: {len(files_after)}")
    print("Sample files after VACUUM:")
    pprint.pprint(files_after[:10])

    # Compute removed files
    removed_files = set(files_before) - set(files_after)
    print(f"\n===== Removed Files by VACUUM: {len(removed_files)} =====")
    if removed_files:
        print("List of removed files:")
        list_removed_files = list(removed_files)[:10]
        pprint.pprint(list_removed_files)
    else:
        print("No files were removed by VACUUM.\n")

In [ ]:
location_raw = f"s3a://staging"
file = "hotel_booking.csv"
data_origen = f"{location_raw}/{file}"

In [ ]:
df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(data_origen)
df = df.withColumnRenamed("phone-number", "phone_number")
df = build_arrival_partition_date(df)

In [ ]:
df.select(
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_day_of_month",
    "arrival_partition_date"
).limit(10).show(truncate=False)

In [ ]:
df.printSchema()

### 2 - Write data into bronze

In [ ]:
table_bronze = "hotel_booking_bronze_vaccum"
table_prefix = f"delta/{table_bronze}"
location_bronze = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze}"

In [ ]:
%%time
(
    df.write.format("delta")
    .mode("overwrite")
    .partitionBy("arrival_partition_date")
    .save(location_bronze)
)

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze}
    USING DELTA
    LOCATION '{location_bronze}'
""")

## 3 - Changing TBLPROPERTIES for the delta table and controlling Vaccum behavior

In [ ]:
delta_table = DeltaTable.forName(spark, f"{DATABASE}.{table_bronze}")

- **delta.logRetentionDuration** defaults to interval 30 days and keeps track of
the history of the table. The more operations that occur, the more history that is
retained. If you won’t be using time travel operations, then you can try reducing
the number of days of history down to a week.

- **delta.deletedFileRetentionDuration** defaults to interval 1 week and can be
changed in cases where delete operations are not expected to be undone. For
peace of mind, it is good to maintain at least one day for deleted files to be
retained.

fonte

_Delta Lake: The definetive Guide_

In [ ]:
alter_table_retention_sql = f"""
ALTER TABLE {DATABASE}.{table_bronze}
SET TBLPROPERTIES (
    delta.logRetentionDuration = 'interval 30 days'
)
"""

spark.sql(alter_table_retention_sql)

In [ ]:
alter_table_ret_deleted_sql = f"""
ALTER TABLE {DATABASE}.{table_bronze}
SET TBLPROPERTIES (
    delta.deletedFileRetentionDuration = 'interval 1 week'
)
"""

spark.sql(alter_table_ret_deleted_sql)

In [ ]:
# Usando SQL para descrever detalhes da tabela Delta
detail_df = spark.sql(f"DESCRIBE DETAIL {DATABASE}.{table_bronze}")
detail_df.select("properties").show(truncate=False)

## 4 - Operation: **Update**

In [ ]:
%%time
# Update 'lead_time' by incrementing by 1 where 'is_canceled' is 0
delta_table.update(
    condition = "lead_time > 100 AND reservation_status = 'Check-Out'",
    set = { "lead_time": "lead_time + 1" }
)
print("Records updated successfully to create new versions.")

## 5 - Apply Vaccum in the Delta Table WITH TBLPROPERTIES

In [ ]:
%%time
RETENTION_HOURS = 168

# Apply VACUUM and list files before and after
apply_vacuum_and_list_files(
    bucket_name=BUCKET_BRONZE,
    table_prefix=table_prefix,
    retention_hours=RETENTION_HOURS,
)

## 6 - Apply Vaccum in the Delta Table **WITHOUT TBLPROPERTIES**

In [ ]:
alter_table_retention_sql = f"""
ALTER TABLE {DATABASE}.{table_bronze}
SET TBLPROPERTIES (
    delta.logRetentionDuration = 'interval 0 hours'
)
"""

spark.sql(alter_table_retention_sql)

alter_table_ret_deleted_sql = f"""
ALTER TABLE {DATABASE}.{table_bronze}
SET TBLPROPERTIES (
    delta.deletedFileRetentionDuration = 'interval 0 hours'
)
"""

spark.sql(alter_table_ret_deleted_sql)

In [ ]:
%%time
RETENTION_HOURS = 0

# Apply VACUUM and list files before and after
apply_vacuum_and_list_files(
    bucket_name=BUCKET_BRONZE,
    table_prefix=table_prefix,
    retention_hours=RETENTION_HOURS,
)

In [ ]:
spark.stop()